# Lab01 — дополнительное задание КТвД
## 🎧 Танцевальность и настроение музыки

**Вопрос:** можно ли по танцевальности трека предугадать позитивность его звучания?

Два случайных признака одной записи: $X=\mathrm{danceability}$, $Y=\mathrm{valence}$, оба в диапазоне $[0,1]$.

**Навигация:** [Описание](#description) · [EDA](#eda) · [Независимость](#dependence) · [Генератор](#generator) · [Итоги](#resume).

В папке также лежат готовый иллюстрированный отчёт `report.html` и интерактивное приложение `marimo_app.py`.


<a id="description"></a>
## 1. Description — почему это случайные величины

Случайный эксперимент: **равновероятно выбрать один трек** из конечного набора. При фиксированном треке значения X и Y детерминированы, но исход выбора неизвестен, и значения меняются от опыта к опыту.

Из 114 000 строк исходного Spotify-файла оставлены 89 740 различных `track_id` с полными допустимыми признаками. Из них случайно **без возвращения** отобраны 12 000 строк с seed 20261121. Анализ ниже относится к этому набору, а не ко всем песням мира.

Источник: [зеркало Spotify Tracks Dataset, CC0](https://github.com/sai-chaitanya-reddy/spotify-tracks-dataset). Дедупликация нужна: одинаковые песни встречались в разных жанрах.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from generators import conditional_generator, independent_generator
from analysis import spearman_permutation, spearman_interval

df = pd.read_csv('data/spotify_tracks_sample.csv')
assert len(df) == 12000 and df.track_id.is_unique
df[['danceability', 'valence']].describe().round(4)

<a id="eda"></a>
## 2. EDA — описательные статистики

| Показатель | Танцевальность X | Позитивность Y |
|---|---:|---:|
| Среднее | **0,5614** | **0,4689** |
| Медиана | 0,5780 | 0,4570 |
| SD | 0,1780 | 0,2631 |
| N | 12 000 | 12 000 |

У обеих характеристик исходный диапазон $[0,1]$, поэтому дополнительные min–max преобразования не нужны. Если требуется стандартизация, используем $Z=(X-\bar X)/s_X$. Это меняет шкалу, но не превращает распределение в нормальное.

![Гистограммы](figures/01_eda.png)


In [ ]:
df[['danceability','valence']].agg(['min','max','mean','median','std']).round(4)

<a id="dependence"></a>
## 3. Проверка зависимости

**Гипотеза 1:** $H_0:\rho_s=0$, $H_1:\rho_s\ne0$. Перестановочный тест по ранговой корреляции Спирмена, 1999 перестановок, $\alpha=0{,}05$.

**Результат:** $\rho_s\approx0{,}4803$, $p=0{,}0005$, 95% bootstrap CI $[0{,}4664;0{,}4952]$. Нулевая гипотеза отвергается.

**Гипотеза 2:** независимость категорий X и Y после разбиения каждого признака на 5 квантильных групп; критерий $\chi^2$.

**Результат:** $\chi^2=2876{,}193$, df=16, $V$ Крамера $=0{,}2448$; p очень мало, независимость категорий отвергается. Если исходные величины независимы, независимы и их группировки. Поэтому наблюдаем статистическое свидетельство зависимости X и Y.

![График зависимости](figures/02_dependence.png)


In [ ]:
x = df.danceability.to_numpy()
y = df.valence.to_numpy()
print('Spearman rho:', round(stats.spearmanr(x, y).statistic, 4))
print('Permutation p:', spearman_permutation(x, y))
print('Bootstrap 95% CI:', spearman_interval(x, y))
ct = pd.crosstab(pd.qcut(x, 5), pd.qcut(y, 5))
chi = stats.chi2_contingency(ct)
print('Chi-square:', round(chi.statistic, 3), 'df:', chi.dof, 'p:', chi.pvalue)

<a id="generator"></a>
## 4. Генератор — квантильный метод и условная суперпозиция

Для упорядоченной выборки $x_{(1)}\le\dots\le x_{(m)}$ и $U_1\sim U(0,1)$:

$$X^*=\widehat F_X^{-1}(U_1)=x_{(\lfloor mU_1\rfloor+1)}.$$

Далее по квантилям X определяем группу $j=B(X^*)$ и генерируем

$$Y^*=\widehat F^{-1}_{Y\mid B(X)=j}(U_2),\qquad U_2\sim U(0,1),$$

где $U_1$ и $U_2$ независимы. Так сохраняется эмпирическое распределение X и условное распределение Y в квантильных группах. Контрольный генератор получает X и Y независимо и теряет зависимость.

**Проверка:** из 12 000 записей 9600 идут на настройку генератора, 2400 — отложенные. Ниже сравниваются статистики отложенных и синтетических наборов.

![Сравнение генераторов](figures/03_generator.png)


In [ ]:
train = df.sample(frac=.8, random_state=37)
holdout = df.drop(train.index)
synthetic = conditional_generator(train, len(holdout), seed=404, bins=5)
control = independent_generator(train, len(holdout), seed=404)

rows = []
for name, arr in [('train', train), ('holdout', holdout),
                  ('conditional', synthetic), ('independent', control)]:
    rows.append({'Набор': name, 'N': len(arr),
                 'Среднее X': arr.danceability.mean(), 'Среднее Y': arr.valence.mean(),
                 'rho': stats.spearmanr(arr.danceability, arr.valence).statistic})
pd.DataFrame(rows).round(4)

<a id="resume"></a>
## 5. Lab resume — выводы

1. При случайном выборе песни танцевальность и позитивность являются случайными величинами.
2. Для 12 000 треков получена умеренная положительная ранговая зависимость ($\rho_s\approx0{,}48$).
3. Статистические критерии отвергают отсутствие связи на уровне $\alpha=0{,}05$, но не устанавливают причинность.
4. Условный эмпирический генератор сохраняет связь ($\rho_s\approx0{,}462$), независимый — нет ($\rho_s\approx-0{,}002$).
5. В приложении **Marimo** можно менять жанр, объём, seed и число групп, наблюдая эффект на графиках и метриках.

**Ограничение:** генератор имитирует пары числовых характеристик, но не создаёт новую музыку. Источник не является случайной выборкой всех песен мира.

Ссылки на материалы и алгоритмы — в `REPORT.md`, автоматические тесты — `test_generators.py`.
